In [1]:
!pip install pymupdf
!pip install langchain
!pip install langchain-openai
!pip install langchain-chroma
!pip install chromadb
!pip install python-dotenv
!pip install streamlit


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
print(api_key is not None)

True


In [3]:
import fitz

doc = fitz.open("data/sample1.pdf")

text = ""

for page in doc:
    text += page.get_text()

print(text[:2000])

ePublishing System
Published Date: 28-Sep-2026
Basic Details
Organisation Chain
Indian Rare Earths Limited
Tender Reference
Number
GEM/2026/B/7653569
Tender ID
2026_IRELD_841521_1
Withdrawal Allowed
Yes
Tender Type
Open Tender
Form Of Contract
Buy
Tender Category
Goods
No. of Covers
2
General Tech Evaluation
No
ItemWise Tech
Evaluation
No
Payment Mode
Offline
Multi Currency Allowed
(BOQ)
No
Multi Currency Allowed
(Fee)
No
Two Stage Bidding
No
Payment Instruments
Offline
S.No
Instrument Type
1
RTGS / ECS / NEFT
2
As Per Tender Document
Covers Information (No. Of Covers - 2)
Cover
No
Cover Type
Description
Type
1
Fee/PreQual/Technical
Procurement of
LED Street
Light
.pdf
2
Finance
Procurement of
LED Street
Light
.xls
Notice: Document download date is over. Documents can no longer be downloaded.
Fee Details
Tender Fee Details [Total Fee in ₹: 1]
Tender Fee in ₹
1
Fee Payable To
ICAR Unit CSSRI
Fee Payable At
Karnal
Tender Fee
Exemption Allowed
Yes
EMD Fee Details
EMD Amount in ₹
25,025
EM

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\yadav\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_text(text)

print("Total chunks:", len(chunks))

Total chunks: 3


In [6]:
print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- CHUNK {i+1} ---\n")
    print(chunk[:500])

Number of chunks: 3

--- CHUNK 1 ---

ePublishing System
Published Date: 28-Sep-2026
Basic Details
Organisation Chain
Indian Rare Earths Limited
Tender Reference
Number
GEM/2026/B/7653569
Tender ID
2026_IRELD_841521_1
Withdrawal Allowed
Yes
Tender Type
Open Tender
Form Of Contract
Buy
Tender Category
Goods
No. of Covers
2
General Tech Evaluation
No
ItemWise Tech
Evaluation
No
Payment Mode
Offline
Multi Currency Allowed
(BOQ)
No
Multi Currency Allowed
(Fee)
No
Two Stage Bidding
No
Payment Instruments
Offline
S.No
Instrument Type
1
RT

--- CHUNK 2 ---

Fee Details
Tender Fee Details [Total Fee in ₹: 1]
Tender Fee in ₹
1
Fee Payable To
ICAR Unit CSSRI
Fee Payable At
Karnal
Tender Fee
Exemption Allowed
Yes
EMD Fee Details
EMD Amount in ₹
25,025
EMD Exemption
Allowed
Yes
EMD Fee Type
Fixed
EMD Percentage
NA
EMD Payable To
ICAR Unit CSSRI
EMD Payable At
Karnal
ePublishing System, Government of India
Page 1 of 2
Government of India
Work Item Details
Title
Procurement of LED Street Light
Work 

In [7]:
import sys

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)

Python version:
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]

Python executable:
c:\Users\yadav\AppData\Local\Programs\Python\Python312\python.exe


In [8]:
from sentence_transformers import SentenceTransformer

In [9]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6234.86it/s]


Embedding model loaded!


In [10]:
test_embedding = embedding_model.encode(chunks[0])

print("Embedding length:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Embedding length: 384
First 10 values: [-0.05072634  0.02223971 -0.01420557  0.02708231 -0.01188685 -0.04655268
 -0.02871464  0.0809577  -0.0460754   0.02890235]


In [11]:
import chromadb

In [12]:
client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="pdf_documents"
)

print("ChromaDB collection created!")

ChromaDB collection created!


In [13]:
chunk_embeddings = embedding_model.encode(chunks)

print("Number of embeddings:", len(chunk_embeddings))
print("Embedding size:", len(chunk_embeddings[0]))

Number of embeddings: 3
Embedding size: 384


In [14]:
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings.tolist()
)

print("Chunks stored successfully!")

Chunks stored successfully!


In [15]:
query = "What is the tender reference number?"

In [16]:
query_embedding = embedding_model.encode(query).tolist()

In [17]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

In [18]:
for i, document in enumerate(results["documents"][0]):
    print(f"\n--- Retrieved Chunk {i+1} ---")
    print(document)


--- Retrieved Chunk 1 ---
Fee Details
Tender Fee Details [Total Fee in ₹: 1]
Tender Fee in ₹
1
Fee Payable To
ICAR Unit CSSRI
Fee Payable At
Karnal
Tender Fee
Exemption Allowed
Yes
EMD Fee Details
EMD Amount in ₹
25,025
EMD Exemption
Allowed
Yes
EMD Fee Type
Fixed
EMD Percentage
NA
EMD Payable To
ICAR Unit CSSRI
EMD Payable At
Karnal
ePublishing System, Government of India
Page 1 of 2
Government of India
Work Item Details
Title
Procurement of LED Street Light
Work Description
Procurement of LED Street Light
NDA/Pre
Qualification
As per our tender document
Independent
External
Monitor/Remarks
NA
Tender Value in ₹
14,76,475
Product
Category
Miscellaneous
Goods
Sub
category
NA
Contract Type
Tender
Bid Validity
(Days)
90
Period
Of Work
(Days)
60
Location
Manavalakurichi
Pincode
629252
Pre Bid
Meeting
Place
NA
Pre Bid Meeting
Address
NA
Pre Bid
Meeting Date
NA
Bid
Opening
Place
Manavalakurichi
Allow NDA Tender
No
Allow
Preferential
Bidder
No
Critical Dates
Published Date
13-Jun-2026 11:00 

In [19]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

print("API key loaded:", bool(api_key))
print("Key ending:", api_key[-6:] if api_key else None)

API key loaded: True
Key ending: 6FgqEA


In [20]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import ollama

In [22]:
import ollama

response = ollama.chat(
    model="llama3.2:1b",
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ],
    options={"num_ctx": 1024}
)

print(response["message"]["content"])

Hello.


In [23]:
def ask_pdf(question, n_results=2):

    query_embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

    documents = results["documents"][0]
    context = "\n\n".join(documents)

    prompt = f"""
You are a PDF question-answering assistant.

Answer the question using ONLY the information in the context.

If the answer is not present in the context, say:
"I could not find this information in the document."

Context:
{context}

Question:
{question}

Answer:
"""

    response = ollama.chat(
        model="llama3.2:1b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={"num_ctx": 1024}
    )

    return response["message"]["content"]

In [24]:
answer = ask_pdf("What is the tender reference number?")
print(answer)

The tender reference number is "2026_IRELD_841521_1".


In [25]:
answer = ask_pdf("What is the EMD Amount in ₹?")
print(answer)

I could not find this information in the document.
